In [8]:
import os
from langchain.document_loaders import PyPDFLoader, TextLoader, UnstructuredWordDocumentLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.embeddings import OllamaEmbeddings
from langchain.chat_models import ChatOllama
from langchain.schema import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain.prompts import PromptTemplate

In [9]:
# 📂 Путь к папке с документами
DATA_DIR = "shoulder"
VECTORSTORE_DIR = "faiss_index"

# 🚀 Загрузка документов
def load_documents(directory):
    docs = []
    for file in os.listdir(directory):
        path = os.path.join(directory, file)
        if file.endswith(".pdf"):
            docs.extend(PyPDFLoader(path).load())
        elif file.endswith(".txt"):
            docs.extend(TextLoader(path).load())
    return docs

# ✂️ Разделение документов на чанки
def split_documents(docs):
    splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=200)
    return splitter.split_documents(docs)

# 🧠 Создание векторной базы
def create_vectorstore(chunks):
    embeddings = OllamaEmbeddings(model="nomic-embed-text")
    vectorstore = FAISS.from_documents(chunks, embeddings)
    vectorstore.save_local(VECTORSTORE_DIR)  # ✅ Сохраняем
    return vectorstore

# 🧠 Загрузка векторной базы
def load_or_create_vectorstore():
    if os.path.exists(VECTORSTORE_DIR):
        print("✅ Найдена сохранённая база, загружаем...")
        embeddings = OllamaEmbeddings(model="nomic-embed-text")
        return FAISS.load_local(VECTORSTORE_DIR, embeddings)
    else:
        print("⚡ База не найдена, создаём новую...")
        docs = load_documents(DATA_DIR)
        chunks = split_documents(docs)
        return create_vectorstore(chunks)

# 🔗 Создание RAG цепочки
def create_rag_chain(vectorstore):
    retriever = vectorstore.as_retriever()
    llm = ChatOllama(model="llama3", temperature=0.3)
    return retriever, llm

# 🧩 Формирование промпта с данными пользователя
def create_prompt(user_data, retrieved_docs):
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])

    prompt = f"""
Ты — профессиональный фитнес-тренер.

Данные пользователя:
- Возраст: {user_data['age']}
- Пол: {user_data['gender']}
- Рост: {user_data['height']} см
- Вес: {user_data['weight']} кг
- Частота тренировок в неделю: {user_data['frequency']}
- Оценка физической формы (1-10): {user_data['fitness_level']}
- Желаемый уровень сложности программы: {user_data['program_difficulty']}
- Доступное оборудование/место: {user_data['equipment']}
- Основные группы мышц для тренировки: {user_data['muscle_groups']}

Методические рекомендации:
{context}

‼️ ВАЖНО: Для каждого упражнения обязательно укажи:
- Название упражнения
- Количество подходов
- Количество повторений
- Рекомендуемый вес (если применимо)
- Краткое описание техники выполнения

Структурируй ответ в формате:

Упражнение 1:
- Название:
- Подходы:
- Повторения:
- Вес:
- Техника:

Упражнение 2:
...

Составь план на 2 месяца с тренировками 3 раза в неделю, разнообразие упражнений обязательно (не менее 10 разных на каждую основную группу мышц).
"""
    return prompt

# 🚀 Основной запуск
if __name__ == "__main__":
    # 🔥 Загрузка или создание базы
    vectorstore = load_or_create_vectorstore()

    # 🔗 Создание цепочки
    retriever, llm = create_rag_chain(vectorstore)

    # 📋 Данные пользователя
    user_data = {
        "age": 25,
        "gender": "Мужской",
        "height": 180,
        "weight": 75,
        "frequency": 3,
        "fitness_level": 5,
        "program_difficulty": "Средний",
        "equipment": "Гантели дома, штанга",
        "muscle_groups": "Глечи"
    }
    


    # 🔎 Поиск документов
    query = "Методика составления тренировок и упражнения для групп мышц"
    retrieved_docs = retriever.get_relevant_documents(query)

    # 📝 Генерация промпта
    prompt = create_prompt(user_data, retrieved_docs)

# 🎯 Определим шаблон промпта
prompt_template = PromptTemplate.from_template("{context}")

# 🧠 Модель
model = ChatOllama(model="llama3", temperature=0.3)
    
# 🔗 Цепочка RAG
rag_chain = (
    {
        "context": RunnablePassthrough(),
        "question": RunnablePassthrough(),  # можно игнорировать, если не используешь в шаблоне
    }
    | prompt_template
    | model
    | StrOutputParser()
)

# 📝 Генерация промпта
prompt = create_prompt(user_data, retrieved_docs)

# ⚡ Запуск генерации
response = rag_chain.invoke({"context": prompt, "question": ""})  # Вопрос можно не передавать, он не используется

# 📤 Вывод результата
print(response)


⚡ База не найдена, создаём новую...
Based on the provided context and data, I will create a personalized workout plan for the client. The goal is to improve their physical form, reduce the risk of shoulder injuries, and boost range of movement.

**Month 1**

* **Day 1:**
	+ Упражнение 1: Side Lateral Raise
		- Название: Side Lateral Raise
		- Подходы: 3
		- Повторения: 10
		- Вес: 5 kg (adjustable)
		- Техника: Begin with arms at sides, slowly raise arms to the side until they reach shoulder height, then slowly lower to starting position.
	+ Упражнение 2: Push Ups
		- Название: Push Ups
		- Подходы: 3
		- Повторения: Until failure
		- Вес: Bodyweight
		- Техника: Start in the "knees on floor position" or try the "hands up" style by lying flat on floor and pushing yourself up and controlling your descent back to starting position.
* **Day 2:**
	+ Упражнение 1: Band Pull-Apart (Resistance Band Shoulder Workout)
		- Название: Band Pull-Apart
		- Подходы: 3
		- Повторения: 15
		- Вес: Resi